<a href="https://colab.research.google.com/github/RomanLuja/Python/blob/main/HW_%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%BA%D0%B0_%D0%B3%D1%96%D0%BF%D0%BE%D1%82%D0%B5%D0%B7%D0%B8_SciPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✍️ Практика: перевірка гіпотез і статистичні тести в SciPy

**Самостійна робота · ⏱ 3 години · 37 завдань (з них 7 бонусні)**

Основні дані — **A/B-тест нової сторінки оплати** в інтернет-магазині «Затишок»: 4000 відвідувачів, половина бачила стару сторінку (A), половина — нову (B). Плюс кілька невеликих наборів: час доставки, продажі менеджерів до і після тренінгу, замовлення за днями тижня, реклама і продажі.

| Блок | Тема | Час |
|---|---|---|
| 1 | Гіпотези, помилки, p-value: розминка | 15 хв |
| 2 | p-value: симуляція і точний тест | 20 хв |
| 3 | Розподіли в `scipy.stats` | 25 хв |
| 4 | Описова статистика і довірчі інтервали | 25 хв |
| 5 | t-тест | 30 хв |
| 6 | Критерій хі-квадрат | 25 хв |
| 7 | Кореляція з p-value | 20 хв |
| 8 | Повний аналіз A/B-тесту | 20 хв |

---

### Як працювати з цим блокнотом

1. **Запускайте клітинки по черзі зверху вниз** (Shift + Enter). Спочатку обов'язково запустіть три службові клітинки в розділі «Підготовка».
2. **Пропуски `ВАШ_КОД`** у коді треба замінити на правильний код. Поки пропуск не заповнено, клітинка видасть помилку на кшталт `NameError: name 'ВАШ_КОД' is not defined`, і це нормально. У блоці 1 замініть `'ВАШ_КОД'` на літеру відповіді, наприклад `'б'`.
3. **Підказки відкриваються кліком** на «💡 Підказка 1». Спершу спробуйте самі, потім відкрийте першу підказку. Друга підказка майже містить відповідь, тож користуйтеся нею, якщо зовсім застрягли.
4. **Самоперевірка:** після більшості завдань є клітинка з `check(...)`. Запустіть її, і вона напише ✅ або ❌.
5. **Текстові відповіді:** клітинки «✍️ Ваша відповідь» двічі клацніть і напишіть відповідь своїми словами.
6. Завдання з ⭐ — **бонусні**. Якщо часу мало, їх можна пропустити.
7. Застрягли на завданні більше ніж на 10 хвилин — переходьте до наступного і поверніться пізніше.

> 💡 Відкрити в Google Colab: *File → Upload notebook*. Усі дані генеруються всередині блокнота.

---
## Підготовка

Запустіть три клітинки нижче: бібліотеки, дані та функції самоперевірки.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.float_format', '{:.4f}'.format)

### Дані

**`ab`** — A/B-тест сторінки оплати (4000 рядків):

| Колонка | Що означає |
|---|---|
| `group` | A — стара сторінка, B — нова |
| `device` | Мобільний / Комп'ютер |
| `age` | вік |
| `time_on_site` | хвилин на сайті |
| `converted` | купив (True) чи ні (False) |
| `revenue` | сума покупки, грн (0, якщо не купив) |

**Невеликі набори:** `delivery` — час доставки 35 замовлень, хв · `before`, `after` — продажі 15 менеджерів до і після тренінгу · `weekday_orders` — замовлення за днями Пн…Нд · `ad_spend`, `weekly_sales` — реклама (тис. грн) і продажі за 12 тижнів.

In [3]:
rng = np.random.default_rng(2028)

# --- 1. A/B-тест нової сторінки оплати в інтернет-магазині «Затишок» ---
n = 4000
group = rng.choice(['A', 'B'], size=n)                        # A — стара сторінка, B — нова
device = rng.choice(['Мобільний', "Комп'ютер"], size=n, p=[0.6, 0.4])
age = np.round(np.clip(rng.normal(34, 10, n), 18, 70)).astype(int)
p_buy = 0.09 + 0.025 * (group == 'B') + 0.03 * (device == "Комп'ютер")
converted = rng.random(n) < p_buy

ab = pd.DataFrame({
    'user_id':      np.arange(1, n + 1),
    'group':        group,
    'device':       device,
    'age':          age,
    'time_on_site': np.round(rng.gamma(3, 2.5, n) + 0.03 * age, 1),   # хвилин на сайті
    'converted':    converted,                                         # купив чи ні
    'revenue':      np.where(converted, np.round(rng.lognormal(6.5, 0.45, n)), 0),
})

# --- 2. Час доставки 35 випадкових замовлень, хв ---
delivery = np.round(rng.normal(35, 7, 35), 1)

# --- 3. Продажі 15 менеджерів за місяць до і після тренінгу ---
before = np.round(rng.normal(48, 9, 15)).astype(int)
after = before + np.round(rng.normal(3.5, 4, 15)).astype(int)

# --- 4. Кількість замовлень за днями тижня (Пн … Нд) ---
weekday_orders = np.array([118, 112, 125, 131, 156, 182, 146])

# --- 5. Реклама і продажі за 12 тижнів ---
ad_spend = np.round(rng.uniform(10, 60, 12), 1)              # тис. грн
weekly_sales = np.round(150 + 4 * ad_spend + rng.normal(0, 30, 12))

print('ab:', ab.shape, '| delivery:', delivery.shape, '| before/after:', before.shape,
      '| weekday_orders:', weekday_orders.shape, '| ad_spend:', ad_spend.shape)

ab: (4000, 7) | delivery: (35,) | before/after: (15,) | weekday_orders: (7,) | ad_spend: (12,)


In [4]:
# ⚙️ Службова клітинка: функції для самоперевірки. Просто запустіть її, змінювати нічого не потрібно.
ANSWERS = {
    '1.1': ['б'],
    '1.2': ['б', 'а'],
    '1.3': [1.0, 2.0],
    '1.4': ['б'],
    '2.1': [0.020979],
    '2.2': [0.020979, 1.0],
    '2.3': [0.368202, 0.0],
    '2.4': [0.010489],
    '3.1': [0.02275, 0.058042],
    '3.2': [184.970861],
    '3.3': [1.959964, 2.575829],
    '3.4': [0.28518, 0.323073],
    '3.5': [0.104196, 0.021363],
    '3.6': [2.262157],
    '4.1': [35.0, 33.482857, 37.206756],
    '4.2': [1.031043, 1.031043],
    '4.3': [[31.387525, 35.578189]],
    '4.4': [4.190664, 5.626185],
    '4.5': [0.100601, [0.087411, 0.11379]],
    '4.6': [[31.0, 35.1]],
    '5.1': [3.377993, 0.001844, 1.0],
    '5.2': [-35.280339, 0.268799],
    '5.3': [1.4, 0.019866],
    '5.4': [0.772331],
    '5.5': [0.000922],
    '6.1': [0.100601, 0.123377],
    '6.2': [4.989349, 0.025504, 1.0],
    '6.3': [0.007068, "Комп'ютер"],
    '6.4': [0.000214],
    '6.5': [175.28],
    '7.1': [0.950888, 2e-06],
    '7.2': [0.937063],
    '7.3': [0.066948, 2.3e-05],
    '7.4': [0.674285, 0.066659],
    '8.1': [0.022776, 2.283822, 0.022382, [0.003244, 0.042308]],
    '8.3': [0.05],
}
TOL = {'2.1': 0.01}

def _same(v, e, tol=None):
    if isinstance(e, str):
        latin = {'a': 'а', 'b': 'б', 'v': 'в', 'c': 'в'}       # латинські літери замість кириличних теж зараховуємо
        v = str(v).strip().lower()
        return latin.get(v, v) == e.lower()
    if isinstance(e, list):
        try:
            v = np.asarray(v, dtype=float)
            return v.shape == np.asarray(e).shape and np.allclose(v, e, atol=0.01, rtol=0.005)
        except (TypeError, ValueError):
            return False
    try:
        return abs(float(v) - e) <= (tol if tol is not None else max(0.0005, abs(e) * 0.005))
    except (TypeError, ValueError):
        return False

def check(task, *values):
    expected = ANSWERS[task]
    if len(values) != len(expected):
        print(f'⚠️ Очікую {len(expected)} значення(нь), а отримала функція {len(values)}.')
        return
    if all(_same(v, e, TOL.get(task)) for v, e in zip(values, expected)):
        print(f'✅ Завдання {task}: правильно! Можна рухатися далі.')
    else:
        print(f'❌ Завдання {task}: поки не збігається. Перевірте код і відкрийте підказку.')

---

## Блок 1. Гіпотези, помилки і p-value: розминка

Тут не треба писати код — лише обрати правильну відповідь і записати її літеру в змінну. Автоперевірка підкаже, чи ви праві.

**Нагадування:**
- **H₀** — «ефекту немає», «різниці немає», «нічого не змінилось».
- **H₁** — те, що ми хочемо показати: «різниця є».
- **Двостороння** H₁: «відрізняється» (≠). **Одностороння**: «більше» (>) або «менше» (<).
- **Помилка I роду** — відхилили H₀, хоча вона правдива (хибна тривога). **Помилка II роду** — не відхилили H₀, хоча вона хибна (пропустили ефект).
- **p-value** — ймовірність отримати такі або ще екстремальніші дані, **якщо H₀ правдива**.

### Завдання 1.1. Яка нульова гіпотеза?

Магазин перевіряє, чи **нова сторінка оплати змінює конверсію**. Що є H₀?

- **а)** нова сторінка дає вищу конверсію
- **б)** конверсії на старій і новій сторінці однакові
- **в)** конверсія нової сторінки дорівнює 100%

<details>
<summary><b>💡 Підказка 1</b></summary>

H₀ завжди описує ситуацію «нічого не змінилось», «ефекту немає».

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

Варіант «а» — це H₁ (те, що хочемо показати). Варіант «в» — взагалі не гіпотеза про порівняння.

</details>

In [11]:
answer_1_1 = "б"  # впишіть літеру: а, б або в

In [12]:
check('1.1', answer_1_1)

✅ Завдання 1.1: правильно! Можна рухатися далі.


### Завдання 1.2. Одностороння чи двостороння?

Для кожної ситуації оберіть тип гіпотези: **а** — одностороння, **б** — двостороння.

1. Ми хочемо дізнатися, чи середній час доставки **відрізняється** від обіцяних 30 хв (неважливо, в який бік).
2. Фармкомпанія хоче показати, що нові ліки **знижують** тиск (підвищення тиску її не цікавить).

<details>
<summary><b>💡 Підказка 1</b></summary>

Ключові слова: «відрізняється» — у будь-який бік; «знижують», «покращує», «більше» — лише в один бік.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

Перша ситуація — «≠», друга — «<».

</details>

In [15]:
answer_1_2_first =  'б'     # а або б
answer_1_2_second = 'а'    # а абоc

In [16]:
check('1.2', answer_1_2_first, answer_1_2_second)

✅ Завдання 1.2: правильно! Можна рухатися далі.


### Завдання 1.3. Помилка якого роду?

Запишіть **1** (помилка I роду) або **2** (помилка II роду):

1. Ми запустили нову кнопку, бо тест показав «покращення», а насправді вона нічим не краща за стару.
2. Нова кнопка насправді краща, але тест цього не показав, і ми від неї відмовились.

<details>
<summary><b>💡 Підказка 1</b></summary>

Помилка I роду — «хибна тривога»: побачили ефект, якого немає. Помилка II роду — «пропустили» справжній ефект.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

У першому випадку ми **відхилили** H₀ («кнопки однакові»), хоча вона правдива. У другому — **не відхилили** H₀, хоча вона хибна.

</details>

In [17]:
error_case_1 = 1    # 1 або 2
error_case_2 = 2   # 1 або 2

In [18]:
check('1.3', error_case_1, error_case_2)

✅ Завдання 1.3: правильно! Можна рухатися далі.


### Завдання 1.4. Що означає p = 0.03?

A/B-тест дав **p-value = 0.03**. Що це означає?

- **а)** ймовірність, що H₀ правдива, дорівнює 3%
- **б)** якщо H₀ правдива, такі або ще більші відмінності між групами трапляються приблизно у 3% випадків
- **в)** нова версія краща на 3%

<details>
<summary><b>💡 Підказка 1</b></summary>

p-value рахують, **припускаючи**, що H₀ правдива. Тому він не може бути «ймовірністю, що H₀ правдива».

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

p-value — це про **дані** за умови H₀, а не про гіпотезу і не про розмір ефекту.

</details>

In [32]:
answer_1_4 = 'б'

In [33]:
check('1.4', answer_1_4)

✅ Завдання 1.4: правильно! Можна рухатися далі.


---

## Блок 2. p-value: симуляція і точний тест

**Ситуація:** монетку підкинули 100 разів, випало **62 орли**. H₀: монетка чесна (ймовірність орла 0.5). H₁: монетка нечесна. α = 0.05.

**Нагадування:**
- `gen.binomial(n, p, size=k)` — k разів «підкинути» монетку n разів; повертає кількість орлів у кожному експерименті.
- `stats.binomtest(k, n=..., p=...)` — точний тест; p-value лежить у `.pvalue`.
- Рішення: `p_value <= 0.05` → відхиляємо H₀.

### Завдання 2.1. p-value симуляцією

Змоделюйте 100 000 експериментів з **чесною** монеткою по 100 кидків. Порахуйте частку експериментів, де кількість орлів відхиляється від 50 **так само сильно або сильніше**, ніж у нас (|орли − 50| ≥ 12). Це і є p-value. Збережіть у `p_sim`.

<details>
<summary><b>💡 Підказка 1</b></summary>

Відхилення в нашому випадку: |62 − 50| = 12. Умову «так само далеко або далі» записують через `np.abs(sim - 50) >= 12`. Частку True дає `.mean()`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
sim = gen.binomial(100, 0.5, size=100_000)
p_sim = (np.abs(sim - 50) >= 12).mean()
```

</details>

In [34]:
gen = np.random.default_rng(1)
sim = gen.binomial(100, 0.5, size=100_000)

p_sim = (np.abs(sim - 50) >= 12).mean()
print(f'p-value (симуляція): {p_sim:.4f}')

p-value (симуляція): 0.0208


In [35]:
check('2.1', p_sim)

✅ Завдання 2.1: правильно! Можна рухатися далі.


### Завдання 2.2. Точний біноміальний тест

Порахуйте точне p-value для 62 орлів зі 100 функцією `stats.binomtest` (`p_exact`). Збережіть у `reject_62` рішення: `True`, якщо відхиляємо H₀ при α = 0.05.

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.binomtest(62, n=100, p=0.5)` повертає об'єкт, p-value — у атрибуті `.pvalue`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
p_exact = stats.binomtest(62, n=100, p=0.5).pvalue
reject_62 = p_exact <= 0.05
```

</details>

In [36]:
p_exact = stats.binomtest(62, n=100, p=0.5).pvalue
reject_62 = p_exact <= 12
print(f'p-value: {p_exact:.4f} | відхиляємо H₀? {reject_62}')

p-value: 0.0210 | відхиляємо H₀? True


In [37]:
check('2.2', p_exact, reject_62)

✅ Завдання 2.2: правильно! Можна рухатися далі.


**✍️ Сформулюйте висновок одним-двома реченнями простою мовою.**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 2.3. А якщо 55 орлів?

Повторіть тест для 55 орлів зі 100 (`p_55`, `reject_55`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Скопіюйте код із 2.2 і замініть 62 на 55.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
p_55 = stats.binomtest(55, n=100, p=0.5).pvalue
reject_55 = p_55 <= 0.05
```

</details>

In [ ]:
p_55 = ВАШ_КОД
reject_55 = ВАШ_КОД
print(f'p-value: {p_55:.4f} | відхиляємо H₀? {reject_55}')

In [ ]:
check('2.3', p_55, reject_55)

**✍️ Чи означає результат, що монетка **точно чесна**?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 2.4. Одностороння гіпотеза ⭐ *бонус*

Припустімо, ще **до** експерименту ми підозрювали, що монетка частіше падає **орлом** (H₁: p > 0.5). Порахуйте одностороннє p-value для 62 орлів (`p_greater`) і порівняйте з двостороннім.

<details>
<summary><b>💡 Підказка 1</b></summary>

У `binomtest` є параметр `alternative`: `'two-sided'` (за замовчуванням), `'greater'` або `'less'`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
p_greater = stats.binomtest(62, n=100, p=0.5, alternative='greater').pvalue
```

</details>

In [ ]:
p_greater = stats.binomtest(62, n=100, p=0.5, alternative='ВАШ_КОД').pvalue
print(f'Одностороннє: {p_greater:.4f} | Двостороннє: {p_exact:.4f}')

In [ ]:
check('2.4', p_greater)

---

## Блок 3. Розподіли в `scipy.stats`

| Метод | Що повертає |
|---|---|
| `.cdf(x)` | P(X ≤ x) |
| `.sf(x)` | P(X > x) = 1 − cdf(x) |
| `.ppf(q)` | значення, нижче якого лежить частка q (зворотне до cdf) |
| `.pmf(k)` | ймовірність **рівно** k (лише для дискретних: `binom`, `poisson`) |

Розподіл створюють так: `stats.norm(loc=середнє, scale=станд_відхилення)`, `stats.binom(n=..., p=...)`, `stats.poisson(mu=...)`.

### Завдання 3.1. Зріст: частка високих і низьких

Зріст чоловіків розподілений нормально: μ = 176 см, σ = 7 см. Знайдіть частку чоловіків, **вищих за 190 см** (`p_tall`), і частку **нижчих за 165 см** (`p_short`).

<details>
<summary><b>💡 Підказка 1</b></summary>

«Вищі за 190» — це P(X > 190), метод `.sf()`. «Нижчі за 165» — P(X ≤ 165), метод `.cdf()`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
height = stats.norm(loc=176, scale=7)
p_tall = height.sf(190)
p_short = height.cdf(165)
```

</details>

In [ ]:
height = stats.norm(loc=ВАШ_КОД, scale=ВАШ_КОД)
p_tall = height.ВАШ_КОД(190)
p_short = height.ВАШ_КОД(165)
print(f'Вищі за 190: {p_tall:.2%} | Нижчі за 165: {p_short:.2%}')

In [ ]:
check('3.1', p_tall, p_short)

### Завдання 3.2. Поріг для 10% найвищих

Магазин одягу хоче шити окрему лінію для 10% найвищих чоловіків. Від якого зросту починаються ці 10%? Збережіть у `tall_threshold`.

<details>
<summary><b>💡 Підказка 1</b></summary>

10% найвищих — це все, що вище 90-го перцентиля. Перцентиль шукає метод `.ppf(частка)`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
tall_threshold = height.ppf(0.90)
```

</details>

In [ ]:
tall_threshold = height.ppf(ВАШ_КОД)
print(f'10% найвищих: від {tall_threshold:.1f} см')

In [ ]:
check('3.2', tall_threshold)

### Завдання 3.3. Звідки 1.96?

Знайдіть критичні значення стандартного нормального розподілу для двостороннього 95% (`z_95`) і 99% (`z_99`) довірчого інтервалу.

<details>
<summary><b>💡 Підказка 1</b></summary>

Для 95% поза інтервалом лишається 5%: по 2.5% з кожного боку. Тому верхня межа — це ppf(0.975). Для 99% — по 0.5% з кожного боку.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
z_95 = stats.norm.ppf(0.975)
z_99 = stats.norm.ppf(0.995)
```

</details>

In [ ]:
z_95 = stats.norm.ppf(ВАШ_КОД)
z_99 = stats.norm.ppf(ВАШ_КОД)
print(f'95%: {z_95:.3f} | 99%: {z_99:.3f}')

In [ ]:
check('3.3', z_95, z_99)

### Завдання 3.4. Скільки з 20 відвідувачів куплять?

Кожен відвідувач купує з ймовірністю 0.1. На сайт зайшли 20 людей. Знайдіть ймовірність, що куплять **рівно 2** (`p_exactly_2`) і **щонайменше 3** (`p_at_least_3`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Це біноміальний розподіл: `stats.binom(n=20, p=0.1)`. «Рівно» — `.pmf(k)`. «Щонайменше 3» = «більше 2» = `.sf(2)`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
buyers = stats.binom(n=20, p=0.1)
p_exactly_2 = buyers.pmf(2)
p_at_least_3 = buyers.sf(2)
```

</details>

In [ ]:
buyers = stats.binom(n=ВАШ_КОД, p=ВАШ_КОД)
p_exactly_2 = buyers.ВАШ_КОД(2)
p_at_least_3 = buyers.sf(ВАШ_КОД)
print(f'Рівно 2: {p_exactly_2:.3f} | Щонайменше 3: {p_at_least_3:.3f}')

In [ ]:
check('3.4', p_exactly_2, p_at_least_3)

### Завдання 3.5. Дзвінки в підтримку

У середньому в підтримку дзвонять 4 рази на годину (розподіл Пуассона). Знайдіть ймовірність **рівно 6** дзвінків за годину (`p_6`) і **більше 8** (`p_more_8`).

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.poisson(mu=4)`. Методи ті самі: `.pmf(k)` — рівно k, `.sf(k)` — більше k.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
calls = stats.poisson(mu=4)
p_6 = calls.pmf(6)
p_more_8 = calls.sf(8)
```

</details>

In [ ]:
calls = stats.poisson(mu=ВАШ_КОД)
p_6 = ВАШ_КОД
p_more_8 = ВАШ_КОД
print(f'Рівно 6: {p_6:.3f} | Більше 8: {p_more_8:.4f}')

In [ ]:
check('3.5', p_6, p_more_8)

### Завдання 3.6. Критичне значення t-розподілу ⭐ *бонус*

Знайдіть критичне значення t-розподілу для 95% ДІ, якщо у вибірці 10 спостережень (df = 9). Збережіть у `t_crit` і порівняйте з 1.96.

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.t.ppf(0.975, df=9)`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
t_crit = stats.t.ppf(0.975, df=9)
```

</details>

In [ ]:
t_crit = stats.t.ppf(ВАШ_КОД, df=ВАШ_КОД)
print(f't* = {t_crit:.3f} (а для нормального — {z_95:.3f})')

In [ ]:
check('3.6', t_crit)

---

## Блок 4. Описова статистика і довірчі інтервали

Працюємо з `delivery` — час доставки 35 випадкових замовлень.

| Код | Що робить |
|---|---|
| `stats.describe(x)` | n, min/max, середнє, дисперсія, асиметрія, ексцес; доступ: `.nobs`, `.mean`, `.variance` |
| `stats.sem(x)` | стандартна похибка середнього SE = s / √n |
| `stats.t.interval(0.95, df=n-1, loc=x̄, scale=SE)` | 95% ДІ для середнього |
| `stats.norm.interval(0.95, loc=p̂, scale=√(p̂(1−p̂)/n))` | 95% ДІ для частки |

### Завдання 4.1. describe

Застосуйте `stats.describe` до `delivery`. Збережіть кількість спостережень (`n_obs`), середнє (`d_mean`) і дисперсію (`d_var`).

<details>
<summary><b>💡 Підказка 1</b></summary>

`d = stats.describe(delivery)`, а потім поля `d.nobs`, `d.mean`, `d.variance`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
d = stats.describe(delivery)
n_obs, d_mean, d_var = d.nobs, d.mean, d.variance
```

</details>

In [ ]:
d = stats.describe(ВАШ_КОД)
n_obs = d.ВАШ_КОД
d_mean = d.ВАШ_КОД
d_var = d.ВАШ_КОД
print(f'n = {n_obs} | середнє = {d_mean:.2f} | дисперсія = {d_var:.2f}')

In [ ]:
check('4.1', n_obs, d_mean, d_var)

### Завдання 4.2. Стандартна похибка

Порахуйте стандартну похибку середнього через `stats.sem` (`se`) і вручну за формулою s / √n (`se_manual`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Вручну: стандартне відхилення вибірки — `delivery.std(ddof=1)`, корінь — `np.sqrt(n)`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
se = stats.sem(delivery)
se_manual = delivery.std(ddof=1) / np.sqrt(len(delivery))
```

</details>

In [ ]:
se = stats.ВАШ_КОД(delivery)
se_manual = delivery.std(ddof=ВАШ_КОД) / np.sqrt(len(delivery))
print(f'SE (scipy): {se:.4f} | SE (вручну): {se_manual:.4f}')

In [ ]:
check('4.2', se, se_manual)

### Завдання 4.3. 95% довірчий інтервал для середнього

Побудуйте 95% ДІ для середнього часу доставки через `stats.t.interval`. Збережіть у `ci95` (пара чисел: нижня і верхня межа).

<details>
<summary><b>💡 Підказка 1</b></summary>

Потрібні чотири аргументи: рівень довіри 0.95, `df = n − 1`, `loc` = середнє, `scale` = SE.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
ci95 = stats.t.interval(0.95, df=n_obs - 1, loc=d_mean, scale=se)
```

</details>

In [ ]:
ci95 = stats.t.interval(ВАШ_КОД, df=n_obs - ВАШ_КОД, loc=ВАШ_КОД, scale=ВАШ_КОД)
print(f'95% ДІ: {ci95[0]:.2f} … {ci95[1]:.2f} хв')

In [ ]:
check('4.3', ci95)

**✍️ Сервіс обіцяє середню доставку за 30 хв. Що можна сказати, подивившись лише на цей ДІ?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 4.4. 99% інтервал ширший

Побудуйте 99% ДІ (`ci99`) і порівняйте ширину 95% і 99% інтервалів (`width_95`, `width_99`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Змініть 0.95 на 0.99. Ширина інтервалу = верхня межа − нижня.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
ci99 = stats.t.interval(0.99, df=n_obs - 1, loc=d_mean, scale=se)
width_95 = ci95[1] - ci95[0]
width_99 = ci99[1] - ci99[0]
```

</details>

In [ ]:
ci99 = stats.t.interval(ВАШ_КОД, df=n_obs - 1, loc=d_mean, scale=se)
width_95 = ci95[1] - ci95[0]
width_99 = ВАШ_КОД
print(f'99% ДІ: {ci99[0]:.2f} … {ci99[1]:.2f}')
print(f'Ширина 95%: {width_95:.2f} | 99%: {width_99:.2f}')

In [ ]:
check('4.4', width_95, width_99)

### Завдання 4.5. Довірчий інтервал для конверсії

Знайдіть конверсію групи A з таблиці `ab` (`p_hat_a`) і її 95% ДІ (`ci_conv_a`) через нормальне наближення.

<details>
<summary><b>💡 Підказка 1</b></summary>

Конверсія = частка `True` у колонці `converted` серед рядків групи A, тобто `.mean()`. SE частки = √(p̂(1 − p̂) / n).

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
conv_a = ab.loc[ab['group'] == 'A', 'converted']
p_hat_a = conv_a.mean()
se_a = np.sqrt(p_hat_a * (1 - p_hat_a) / len(conv_a))
ci_conv_a = stats.norm.interval(0.95, loc=p_hat_a, scale=se_a)
```

</details>

In [ ]:
conv_a = ab.loc[ab['group'] == 'ВАШ_КОД', 'converted']
p_hat_a = conv_a.ВАШ_КОД()
se_a = np.sqrt(p_hat_a * (1 - p_hat_a) / len(conv_a))
ci_conv_a = stats.norm.interval(0.95, loc=ВАШ_КОД, scale=ВАШ_КОД)
print(f'Конверсія A: {p_hat_a:.2%}, 95% ДІ: {ci_conv_a[0]:.2%} … {ci_conv_a[1]:.2%}')

In [ ]:
check('4.5', p_hat_a, ci_conv_a)

### Завдання 4.6. Бутстреп-ДІ для медіани ⭐ *бонус*

Побудуйте 95% бутстреп-ДІ для **медіани** часу доставки: 5000 разів візьміть з `delivery` вибірку того самого розміру **з поверненням**, порахуйте медіану, а потім 2.5-й і 97.5-й перцентилі цих медіан. Збережіть межі в `ci_median`.

<details>
<summary><b>💡 Підказка 1</b></summary>

Вибірка з поверненням: `gen.choice(delivery, size=len(delivery), replace=True)`. Перцентилі: `np.percentile(масив, [2.5, 97.5])`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
boot = [np.median(gen.choice(delivery, size=len(delivery), replace=True)) for _ in range(5000)]
ci_median = np.percentile(boot, [2.5, 97.5])
```

</details>

In [ ]:
gen = np.random.default_rng(4)
boot = [np.median(gen.choice(delivery, size=len(delivery), replace=ВАШ_КОД)) for _ in range(5000)]
ci_median = np.percentile(boot, [ВАШ_КОД, ВАШ_КОД])
print(f'Медіана: {np.median(delivery):.1f} хв, 95% ДІ: {ci_median[0]:.1f} … {ci_median[1]:.1f}')

In [ ]:
check('4.6', ci_median)

---

## Блок 5. t-тест

| Тест | Питання | Функція |
|---|---|---|
| Одновибірковий | середнє = заданому числу? | `stats.ttest_1samp(x, popmean=...)` |
| Незалежний (Велча) | середні двох **різних** груп однакові? | `stats.ttest_ind(a, b, equal_var=False)` |
| Парний | змінилось у **тих самих** об'єктів? | `stats.ttest_rel(before, after)` |

Усі повертають об'єкт з полями `.statistic` (t) і `.pvalue`.

### Завдання 5.1. Чи дотримується сервіс обіцянки 30 хв?

H₀: середній час доставки = 30 хв; H₁: ≠ 30 хв. Проведіть одновибірковий t-тест. Збережіть t-статистику (`t_1`), p-value (`p_1`) і рішення при α = 0.05 (`reject_1`).

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.ttest_1samp(delivery, popmean=30)`. Рішення: `p_1 <= 0.05`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
res = stats.ttest_1samp(delivery, popmean=30)
t_1, p_1 = res.statistic, res.pvalue
reject_1 = p_1 <= 0.05
```

</details>

In [ ]:
res = stats.ttest_1samp(ВАШ_КОД, popmean=ВАШ_КОД)
t_1 = res.statistic
p_1 = res.ВАШ_КОД
reject_1 = ВАШ_КОД
print(f't = {t_1:.2f}, p-value = {p_1:.5f} | відхиляємо H₀? {reject_1}')

In [ ]:
check('5.1', t_1, p_1, reject_1)

### Завдання 5.2. Чи змінився середній чек?

Нова сторінка оплати могла вплинути не лише на конверсію, а й на суму покупки. Порівняйте `revenue` **покупців** у групах A і B t-тестом Велча. Збережіть різницю середніх B − A (`diff_rev`) і p-value (`p_rev`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Беремо лише тих, хто купив: умова `ab['converted']` плюс умова на групу. Дві умови з'єднуємо `&`, кожну — в дужках.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
rev_a = ab.loc[(ab['group'] == 'A') & ab['converted'], 'revenue']
rev_b = ab.loc[(ab['group'] == 'B') & ab['converted'], 'revenue']
diff_rev = rev_b.mean() - rev_a.mean()
p_rev = stats.ttest_ind(rev_a, rev_b, equal_var=False).pvalue
```

</details>

In [ ]:
rev_a = ab.loc[(ab['group'] == 'A') & ab['converted'], 'revenue']
rev_b = ab.loc[(ab['group'] == 'ВАШ_КОД') & ab['converted'], 'revenue']
print(f'Покупців: A = {len(rev_a)}, B = {len(rev_b)}')
print(f'Середній чек: A = {rev_a.mean():.0f}, B = {rev_b.mean():.0f} грн')

diff_rev = rev_b.mean() - rev_a.ВАШ_КОД()
p_rev = stats.ttest_ind(rev_a, rev_b, equal_var=ВАШ_КОД).pvalue
print(f'Різниця: {diff_rev:+.1f} грн, p-value = {p_rev:.3f}')

In [ ]:
check('5.2', diff_rev, p_rev)

**✍️ Чи впливає нова сторінка на середній чек? Чи можна сказати, що чеки «однакові»?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 5.3. Чи допоміг тренінг?

Порівняйте продажі тих самих 15 менеджерів до (`before`) і після (`after`) тренінгу парним t-тестом. Збережіть середній приріст (`mean_gain`) і p-value (`p_paired`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Дані «до/після» для тих самих людей → `stats.ttest_rel(before, after)`. Приріст кожного — `after - before`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
mean_gain = (after - before).mean()
p_paired = stats.ttest_rel(before, after).pvalue
```

</details>

In [ ]:
print(pd.DataFrame({'до': before, 'після': after, 'приріст': after - before}))

mean_gain = (after - ВАШ_КОД).mean()
p_paired = stats.ttest_rel(ВАШ_КОД, ВАШ_КОД).pvalue
print(f'\nСередній приріст: {mean_gain:+.2f} | p-value = {p_paired:.5f}')

In [ ]:
check('5.3', mean_gain, p_paired)

### Завдання 5.4. Неправильний тест для тих самих даних

Проведіть для `before` і `after` **незалежний** t-тест (як ніби це різні люди). Збережіть p-value у `p_wrong` і порівняйте з `p_paired`.

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.ttest_ind(before, after, equal_var=False)`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
p_wrong = stats.ttest_ind(before, after, equal_var=False).pvalue
```

</details>

In [ ]:
p_wrong = stats.ttest_ind(ВАШ_КОД, ВАШ_КОД, equal_var=False).pvalue
print(f'Парний: p = {p_paired:.5f} | Незалежний: p = {p_wrong:.4f}')

In [ ]:
check('5.4', p_wrong)

**✍️ Чому незалежний тест не помічає ефекту тренінгу, а парний помічає? Який тест тут правильний?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 5.5. Одностороння перевірка ⭐ *бонус*

Менеджер з логістики стверджує, що доставка **довша** за 30 хв (H₁: μ > 30). Проведіть односторонній одновибірковий t-тест. Збережіть p-value у `p_one`.

<details>
<summary><b>💡 Підказка 1</b></summary>

У `ttest_1samp` є параметр `alternative='greater'`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
p_one = stats.ttest_1samp(delivery, popmean=30, alternative='greater').pvalue
```

</details>

In [ ]:
p_one = stats.ttest_1samp(delivery, popmean=30, alternative='ВАШ_КОД').pvalue
print(f'Одностороннє: {p_one:.6f} | Двостороннє: {p_1:.6f}')

In [ ]:
check('5.5', p_one)

---

## Блок 6. Критерій хі-квадрат

| Тест | Питання | Функція |
|---|---|---|
| Незалежність | дві категоріальні ознаки пов'язані? | `chi2, p, dof, expected = stats.chi2_contingency(table)` |
| Згода | частоти відповідають очікуваним? | `stats.chisquare(observed, expected)` |

Таблицю частот для двох колонок зручно отримати через `pd.crosstab(df['колонка1'], df['колонка2'])`.

⚠️ Очікувані частоти в кожній клітинці мають бути ≥ 5.

### Завдання 6.1. Таблиця «група × покупка»

Побудуйте таблицю частот `group` × `converted` (`ct`). Порахуйте конверсію кожної групи (`conv_rate_a`, `conv_rate_b`).

<details>
<summary><b>💡 Підказка 1</b></summary>

`pd.crosstab(ab['group'], ab['converted'])` рахує кількість для кожної пари значень. Конверсію зручно порахувати окремо: `ab.groupby('group')['converted'].mean()`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
ct = pd.crosstab(ab['group'], ab['converted'])
rates = ab.groupby('group')['converted'].mean()
conv_rate_a, conv_rate_b = rates['A'], rates['B']
```

</details>

In [ ]:
ct = pd.crosstab(ab['ВАШ_КОД'], ab['ВАШ_КОД'])
print(ct)

rates = ab.groupby('group')['converted'].ВАШ_КОД()
conv_rate_a = rates['A']
conv_rate_b = rates['ВАШ_КОД']
print(f'\nКонверсія A: {conv_rate_a:.2%} | B: {conv_rate_b:.2%}')

In [ ]:
check('6.1', conv_rate_a, conv_rate_b)

### Завдання 6.2. Хі-квадрат: чи впливає сторінка на покупку?

Застосуйте `stats.chi2_contingency` до таблиці `ct`. Збережіть статистику (`chi2_ab`), p-value (`p_ab`) і рішення (`reject_ab`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Функція повертає 4 значення: статистику, p-value, ступені свободи і таблицю очікуваних частот. Їх можна одразу «розпакувати» в 4 змінні.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
chi2_ab, p_ab, dof, expected = stats.chi2_contingency(ct)
reject_ab = p_ab <= 0.05
```

</details>

In [ ]:
chi2_ab, p_ab, dof, expected = stats.chi2_contingency(ВАШ_КОД)
reject_ab = ВАШ_КОД
print(f'χ² = {chi2_ab:.2f}, p-value = {p_ab:.4f} | відхиляємо H₀? {reject_ab}')
print('\nОчікувані частоти:')
print(pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(1))

In [ ]:
check('6.2', chi2_ab, p_ab, reject_ab)

### Завдання 6.3. Чи пов'язаний пристрій з покупкою?

Перевірте зв'язок `device` × `converted` тим самим способом. Збережіть p-value (`p_dev`) і назву пристрою з вищою конверсією (`best_device`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Побудуйте нову таблицю через `pd.crosstab` і застосуйте `chi2_contingency`. Пристрій з найвищою конверсією — `.idxmax()` від конверсій по пристроях.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
ct_dev = pd.crosstab(ab['device'], ab['converted'])
_, p_dev, _, _ = stats.chi2_contingency(ct_dev)
best_device = ab.groupby('device')['converted'].mean().idxmax()
```

</details>

In [ ]:
ct_dev = pd.crosstab(ab['ВАШ_КОД'], ab['converted'])
print(ct_dev)
_, p_dev, _, _ = stats.chi2_contingency(ВАШ_КОД)

conv_by_device = ab.groupby('device')['converted'].mean()
print(conv_by_device)
best_device = conv_by_device.ВАШ_КОД()
print(f'p-value = {p_dev:.6f} | краща конверсія: {best_device}')

In [ ]:
check('6.3', p_dev, best_device)

### Завдання 6.4. Чи рівномірні замовлення за днями тижня?

Перевірте H₀: замовлення рівномірно розподілені за днями тижня. Очікувана кількість для кожного дня = загальна кількість ÷ 7. Збережіть p-value (`p_week`).

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.chisquare(спостережувані, очікувані)`. Очікувані — масив із 7 однакових чисел: `np.full(7, weekday_orders.sum() / 7)`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
expected_week = np.full(7, weekday_orders.sum() / 7)
p_week = stats.chisquare(weekday_orders, expected_week).pvalue
```

</details>

In [ ]:
expected_week = np.full(7, weekday_orders.sum() / ВАШ_КОД)
print('Очікувано:', expected_week.round(1))
print('Насправді:', weekday_orders)

p_week = stats.chisquare(ВАШ_КОД, ВАШ_КОД).pvalue
print(f'p-value = {p_week:.6f}')

In [ ]:
check('6.4', p_week)

**✍️ Чи рівномірно розподілені замовлення? Що це означає для планування роботи складу?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 6.5. Перевірка умови «очікувані ≥ 5» ⭐ *бонус*

Для таблиці `ct_dev` знайдіть **найменшу** очікувану частоту (`min_expected`). Чи виконується умова застосування хі-квадрат?

<details>
<summary><b>💡 Підказка 1</b></summary>

Таблицю очікуваних частот повертає `chi2_contingency` четвертим значенням. Мінімум масиву — `.min()`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
_, _, _, exp_dev = stats.chi2_contingency(ct_dev)
min_expected = exp_dev.min()
```

</details>

In [ ]:
_, _, _, exp_dev = stats.chi2_contingency(ct_dev)
min_expected = exp_dev.ВАШ_КОД()
print(f'Найменша очікувана частота: {min_expected:.1f} → умова виконується? {min_expected >= 5}')

In [ ]:
check('6.5', min_expected)

---

## Блок 7. Кореляція з p-value

- `r, p = stats.pearsonr(x, y)` — кореляція Пірсона і p-value.
- `rho, p = stats.spearmanr(x, y)` — кореляція Спірмена (за рангами) і p-value.
- **H₀:** кореляції немає (ρ = 0).
- ⚠️ **Значущість ≠ сила.** На великій вибірці навіть дуже слабка кореляція «значуща»; на малій — навіть помітна може бути «незначущою».

### Завдання 7.1. Реклама і продажі

Порахуйте кореляцію Пірсона між `ad_spend` і `weekly_sales` (12 тижнів). Збережіть `r_ad` і `p_ad`.

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.pearsonr(x, y)` повертає два числа, їх можна одразу записати у дві змінні.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
r_ad, p_ad = stats.pearsonr(ad_spend, weekly_sales)
```

</details>

In [ ]:
r_ad, p_ad = stats.pearsonr(ВАШ_КОД, ВАШ_КОД)
print(f'r = {r_ad:.2f}, p-value = {p_ad:.5f}')

plt.scatter(ad_spend, weekly_sales)
plt.title(f'Реклама і продажі: r = {r_ad:.2f}')
plt.xlabel('Реклама, тис. грн')
plt.ylabel('Продажі')
plt.show()

In [ ]:
check('7.1', r_ad, p_ad)

### Завдання 7.2. Кореляція Спірмена

Порахуйте кореляцію Спірмена для тих самих даних (`rho_ad`).

<details>
<summary><b>💡 Підказка 1</b></summary>

`stats.spearmanr(x, y)` — так само повертає два числа.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
rho_ad, p_rho = stats.spearmanr(ad_spend, weekly_sales)
```

</details>

In [ ]:
rho_ad, p_rho = stats.ВАШ_КОД(ad_spend, weekly_sales)
print(f'Спірмен: ρ = {rho_ad:.2f}, p = {p_rho:.5f}')

In [ ]:
check('7.2', rho_ad)

### Завдання 7.3. Значуща, але слабка

У таблиці `ab` (4000 користувачів) перевірте кореляцію між віком (`age`) і часом на сайті (`time_on_site`). Збережіть `r_age` і `p_age`.

<details>
<summary><b>💡 Підказка 1</b></summary>

Ті самі `stats.pearsonr`, тільки з колонками таблиці.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
r_age, p_age = stats.pearsonr(ab['age'], ab['time_on_site'])
```

</details>

In [ ]:
r_age, p_age = stats.pearsonr(ab['ВАШ_КОД'], ab['ВАШ_КОД'])
print(f'r = {r_age:.3f}, p-value = {p_age:.6f}')

In [ ]:
check('7.3', r_age, p_age)

**✍️ Кореляція «статистично значуща». Чи означає це, що вік сильно впливає на час на сайті?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 7.4. Помітна, але незначуща ⭐ *бонус*

Для 8 магазинів дано площу (`area`) і виручку (`revenue_8`). Порахуйте кореляцію і p-value (`r_small`, `p_small`).

<details>
<summary><b>💡 Підказка 1</b></summary>

Знову `stats.pearsonr`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
r_small, p_small = stats.pearsonr(area, revenue_8)
```

</details>

In [ ]:
area = np.array([120, 95, 150, 80, 200, 110, 135, 90])
revenue_8 = np.array([310, 290, 300, 280, 360, 250, 270, 300])

r_small, p_small = stats.pearsonr(ВАШ_КОД, ВАШ_КОД)
print(f'r = {r_small:.2f}, p-value = {p_small:.3f}')

In [ ]:
check('7.4', r_small, p_small)

**✍️ Кореляція помітна, а p-value більший за 0.05 (або близький до межі). Чому так? Що робити аналітику?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

---

## Блок 8. Підсумкове завдання: повний аналіз A/B-тесту

Зберемо все разом. Керівник питає: **«Запускати нову сторінку оплати (B) на всіх?»**

Формула z-тесту для двох часток (з лекції):

$$z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}, \quad \hat{p} = \frac{\text{усі покупки}}{\text{усі відвідувачі}}$$

95% ДІ для різниці: $(\hat{p}_B - \hat{p}_A) \pm 1.96 \cdot \sqrt{\frac{\hat{p}_A(1-\hat{p}_A)}{n_A} + \frac{\hat{p}_B(1-\hat{p}_B)}{n_B}}$

### Завдання 8.1. z-тест і довірчий інтервал для різниці

Порахуйте різницю конверсій B − A (`diff_ab`), z-статистику (`z_ab`), двостороннє p-value (`p_z`) і 95% ДІ для різниці (`ci_diff`). Порівняйте `p_z` з `p_ab` із завдання 6.2: вони трохи різняться, бо `chi2_contingency` для таблиць 2 × 2 за замовчуванням застосовує **поправку Єйтса** (робить тест трохи обережнішим). З параметром `correction=False` p-value збігаються точно.

<details>
<summary><b>💡 Підказка 1</b></summary>

Кількість відвідувачів і покупок у кожній групі можна взяти з таблиці `ct`: `ct.loc['A'].sum()` — усього в A, `ct.loc['A', True]` — купили в A. Двостороннє p-value: `2 * stats.norm.sf(abs(z))`.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
n_a, k_a = ct.loc['A'].sum(), ct.loc['A', True]
n_b, k_b = ct.loc['B'].sum(), ct.loc['B', True]
pa, pb = k_a / n_a, k_b / n_b
diff_ab = pb - pa
p_pool = (k_a + k_b) / (n_a + n_b)
z_ab = diff_ab / np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
p_z = 2 * stats.norm.sf(abs(z_ab))
se_d = np.sqrt(pa * (1 - pa) / n_a + pb * (1 - pb) / n_b)
ci_diff = (diff_ab - 1.96 * se_d, diff_ab + 1.96 * se_d)
```

</details>

In [ ]:
n_a, k_a = ct.loc['A'].sum(), ct.loc['A', True]
n_b, k_b = ct.loc['B'].sum(), ct.loc['B', True]
pa, pb = k_a / n_a, k_b / n_b

diff_ab = ВАШ_КОД - ВАШ_КОД
p_pool = (k_a + k_b) / (n_a + n_b)
z_ab = diff_ab / np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
p_z = 2 * stats.norm.sf(abs(ВАШ_КОД))

se_d = np.sqrt(pa * (1 - pa) / n_a + pb * (1 - pb) / n_b)
ci_diff = (diff_ab - 1.96 * se_d, diff_ab + ВАШ_КОД * se_d)

print(f'A: {pa:.2%} | B: {pb:.2%} | різниця: {diff_ab:+.2%}')
print(f'z = {z_ab:.2f}, p-value = {p_z:.4f}   (хі-квадрат у 6.2: {p_ab:.4f})')
print(f'95% ДІ для різниці: {ci_diff[0]:+.2%} … {ci_diff[1]:+.2%}')

In [ ]:
check('8.1', diff_ab, z_ab, p_z, ci_diff)

### Завдання 8.2. Звіт для керівника

Напишіть висновок з 4–6 речень простою мовою. У ньому мають бути:

1. H₀ і H₁ словами;
2. конверсії груп і різниця (з ДІ);
3. p-value і рішення при α = 0.05;
4. що сталося із середнім чеком (завдання 5.2);
5. рекомендація: запускати чи ні, і що варто відстежувати далі.

<details>
<summary><b>💡 Підказка 1</b></summary>

Перегляньте результати 6.1, 6.2, 5.2 і 8.1. Не пишіть «з ймовірністю 98% B краща» — це неправильна інтерпретація p-value.

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

Шаблон: «Ми перевіряли, чи … (H₀: …). Конверсія B становить …, A — …, різниця … п.п. (95% ДІ: від … до …). p-value = …, тому … H₀. Середній чек … (p = …). Рекомендація: …»

</details>

In [ ]:
# За потреби — додаткові розрахунки

**✍️ Звіт для керівника (4–6 речень):**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

### Завдання 8.3. Пастка множинних порівнянь ⭐ *бонус*

Змоделюйте 200 **A/A-тестів**: обидві групи по 500 людей з однаковою конверсією 10%. Для кожного тесту порахуйте p-value через `chi2_contingency`. Яка частка тестів показала «значущу різницю» (`share_false`)?

<details>
<summary><b>💡 Підказка 1</b></summary>

Для кожного тесту генеруємо кількість покупок у двох групах через `gen.binomial(500, 0.10)` і складаємо таблицю 2 × 2: [[купили_A, не_купили_A], [купили_B, не_купили_B]].

</details>

<details>
<summary><b>💡 Підказка 2 (майже відповідь)</b></summary>

```python
p_list = []
for _ in range(200):
    ka_, kb_ = gen.binomial(500, 0.10), gen.binomial(500, 0.10)
    p_list.append(stats.chi2_contingency([[ka_, 500 - ka_], [kb_, 500 - kb_]])[1])
share_false = (np.array(p_list) <= 0.05).mean()
```

</details>

In [ ]:
gen = np.random.default_rng(8)
p_list = []
for _ in range(200):
    ka_, kb_ = gen.binomial(500, 0.10), gen.binomial(500, 0.10)
    p_list.append(stats.chi2_contingency([[ka_, 500 - ka_], [kb_, 500 - kb_]])[1])

share_false = (np.array(p_list) <= ВАШ_КОД).mean()
print(f'«Значущих» A/A-тестів: {share_false:.1%}')

In [ ]:
check('8.3', share_false)

**✍️ Чому частка «значущих» A/A-тестів не нульова? Що з цього випливає, якщо маркетолог перевіряє 20 різних метрик?**

*Ваша відповідь:* _(двічі клацніть, щоб написати)_

---
## 🎉 Готово!

Ви пройшли повний цикл перевірки гіпотез: від формулювання H₀ і H₁ до t-тестів, хі-квадрат, кореляцій і звіту за результатами A/B-тесту.

**Перед здачею:** *Runtime → Restart and run all* (у Colab) — переконайтеся, що весь блокнот запускається зверху вниз без помилок.